# Data Ingestion — Amazon Product Dataset 2020 (full catalog)

Pipeline: **download (kagglehub) → inspect schema → clean/normalize → embed (all-MiniLM-L6-v2, local) → index (Chroma) → sample hybrid queries**.

Output of this notebook is `data/processed/products.parquet` and a persistent Chroma collection at `data/chroma_db/`, which `src/ingestion/retriever.py` queries and which the `rag.search` MCP tool calls directly.

**Why the whole file and not a single category?** The actual Kaggle file (`marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv`, 10,002 rows) is a general marketplace sample dominated by Toys & Games (6,662 rows) — only 23 rows fall under `Health & Household`, so the "household cleaning" slice the brief suggests doesn't exist here, and `Home & Kitchen` (708 rows) would have thrown away 93% of the catalog to get it. All 10,002 rows are indexed instead, with the top-level category stored as filterable Chroma metadata (`category_top_level`), so `rag.search` can scope a query to one category *at query time* or search across all of them. See the README's "Known data-quality limitations" section for what's missing (`Brand Name`/`Ingredients` are empty across the entire raw file; there's no rating column).

**Does `Category` ever encode multiple categories per product?** Checked in `notebooks/00_eda.ipynb` and documented in `clean.py`'s `_category_top_level` docstring: no. Each row's `Category` is a single `|`-delimited breadcrumb (top → leaf); there's no second delimiter joining independent category assignments, and every `Uniq Id` is unique. Taking the top-level segment is a correct 1:1 extraction, not an approximation.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src" / "ingestion"))

## 1. Download

Requires a Kaggle API token at `~/.kaggle/kaggle.json`. Downloads via `kagglehub` and stages the CSV(s) into `data/raw/`.

In [2]:
from download_data import download

staged_files = download()
staged_files

staged: /Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/raw/marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv (19.6 MB)


[PosixPath('/Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/raw/marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv')]

## 2. Inspect schema

Confirm real column names before trusting the hardcoded `COL_*` constants in `clean.py` — if the team swaps in a different PromptCloud CSV, these may not match.

For this file: `Brand Name`, `Ingredients`, `Asin`, `Sku` all exist as columns but are **entirely empty** (float64/NaN) across all 10,002 rows, and there is **no rating or review-count column at all**. `Model Number` (82%) and `Technical Details` (92%) are well-populated and used elsewhere in the pipeline.

In [3]:
from inspect_schema import inspect

inspect()


=== marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv ===
sampled shape: (500, 28)

columns + dtypes:
  'Uniq Id': object
  'Product Name': object
  'Brand Name': float64
  'Asin': float64
  'Category': object
  'Upc Ean Code': float64
  'List Price': float64
  'Selling Price': object
  'Quantity': float64
  'Model Number': object
  'About Product': object
  'Product Specification': object
  'Technical Details': object
  'Shipping Weight': object
  'Product Dimensions': object
  'Image': object
  'Variants': object
  'Sku': float64
  'Product Url': object
  'Stock': float64
  'Product Details': float64
  'Dimensions': float64
  'Color': float64
  'Ingredients': float64
  'Direction To Use': float64
  'Is Amazon Seller': object
  'Size Quantity Variant': float64
  'Product Description': float64

first row:
{'Uniq Id': '4c69b61db1fc16e7013b43fc926e502d', 'Product Name': 'DB Longboards CoreFlex Crossbow 41" Bamboo Fiberglass Longboard Complete', 'Brand Name': nan

## 3. Clean & normalize

- Reads target fields (`id, title, brand, category, price, ingredients, model_number, url`) from known column names (see README's "Column names" table) — `rating` has no source column and is hardcoded to `None`.
- Parses price out of `$`-prefixed strings only; the raw column also holds garbage like `"from 2 sellers"` or free-text shipping blurbs, and an earlier, unanchored parser mis-read those as fake prices (e.g. `"from 2 sellers"` → `$2.00`). Anchoring on `$` fixes that.
- Derives `category_top_level` from the first segment of the `|`-delimited `Category` breadcrumb. Nothing is filtered out by category — the whole file is kept and the top-level value is carried as a filterable field, so category scoping happens at query time in `rag.search` rather than being baked into the index.
- Adds `brand_inferred`, a deliberately conservative one-word guess from the title, kept in its own column because the real `Brand Name` is 100% empty — never merged into `brand`, so nothing downstream can present a guess as verified data.
- Extracts a `(unit_qty, unit)` from title/weight/description text and derives `price_per_unit` for fair comparisons.
- Builds `features` from `About Product` + `Technical Details` (not `Product Specification`, which turned out to be ~100% boilerplate — `Shipping Weight`/`ASIN`/rank with no spaces between words), stripping recurring boilerplate phrases like "Make sure this fits by entering your model number."
- Writes `data/processed/products.parquet` (16 columns), keyed by a stable `doc_id` used for citations downstream.

In [4]:
from clean import clean

products = clean()
print(products.shape)
products.head()

wrote 10002 products to /Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/processed/products.parquet
(10002, 16)


,id,title,brand,brand_inferred,category,category_top_level,price,rating,ingredients,model_number,url,unit_qty,unit,price_per_unit,features,doc_id
0,4c69b61db1fc16e7013b43fc926e502d,"DB Longboards CoreFlex Crossbow 41"" Bamboo Fib...",NaN,DB,Sports & Outdoors | Outdoor Recreation | Skate...,Sports & Outdoors,237.68,None,NaN,NaN,https://www.amazon.com/DB-Longboards-CoreFlex-...,10.7,lb,22.2131,RESPONSIVE FLEX: The Crossbow features a bambo...,4c69b61db1fc16e7013b43fc926e502d
1,66d49bbed043f5be260fa9f7fbff5957,"Electronic Snap Circuits Mini Kits Classpack, ...",NaN,Electronic,Toys & Games | Learning & Education | Science ...,Toys & Games,99.95,None,NaN,55324,https://www.amazon.com/Electronic-Circuits-Cla...,4.0,lb,24.9875,Snap circuits mini kits classpack provides bas...,66d49bbed043f5be260fa9f7fbff5957
2,2c55cae269aebf53838484b0d7dd931a,3Doodler Create Flexy 3D Printing Filament Ref...,NaN,None,Toys & Games | Arts & Crafts | Craft Kits,Toys & Games,34.99,None,NaN,NaN,https://www.amazon.com/3Doodler-Plastic-Innova...,5.0,ct,6.9980,✅【Smooth 3D drawing experienced the best 3D dr...,2c55cae269aebf53838484b0d7dd931a
3,18018b6bc416dab347b1b7db79994afa,Guillow Airplane Design Studio with Travel Cas...,NaN,Guillow,Toys & Games | Hobbies | Models & Model Kits |...,Toys & Games,28.91,None,NaN,142,https://www.amazon.com/Guillow-Airplane-Design...,13.4,oz,2.1575,Make 8 different Planes at one time. Experimen...,18018b6bc416dab347b1b7db79994afa
4,e04b990e95bf73bbe6a3fa09785d7cd0,Woodstock- Collage 500 pc Puzzle,NaN,Woodstock-,Toys & Games | Puzzles | Jigsaw Puzzles,Toys & Games,17.49,None,NaN,62151,https://www.amazon.com/Woodstock-Collage-500-p...,13.4,oz,1.3052,Puzzle has 500 pieces . Completed puzzle measu...,e04b990e95bf73bbe6a3fa09785d7cd0


In [5]:
products[["price", "rating", "unit_qty", "price_per_unit"]].describe()

,price,unit_qty,price_per_unit
count,9839.000000,8992.000000,8838.000000
mean,39.633625,21.357118,11.985882
std,132.908387,894.250183,73.389444
min,0.010000,0.000000,0.000300
25%,9.990000,1.500000,2.071675
50%,16.990000,3.700000,5.136250
75%,29.990000,8.400000,12.844400
max,5332.000000,83982.000000,5015.500000


## 4. Embed + build the Chroma index

Embedding text = `title + features`. `ingredients` is 100% empty in this file, so it would contribute nothing to the vector and is left out of the embedding text — it's still carried in metadata in case a future dataset populates it. Metadata (`price`, `rating`, `brand`, `category_top_level`, `ingredients`, `model_number`, `price_per_unit` + `unit`, `doc_id`) is stored alongside each vector so retrieval can combine semantic similarity with metadata filters (e.g. `price <= 40`, or scoping to one category). The collection uses cosine distance (`hnsw:space="cosine"`) so the returned `score` is a bounded, interpretable cosine similarity rather than raw L2 distance.

Embedding backend is `all-MiniLM-L6-v2` via `sentence-transformers` (`embeddings.py`) — runs fully local (CPU/MPS/CUDA), no API key. 384-dim vectors, embeds all 10,002 products in well under a minute. Override with `EMBEDDING_MODEL` in `.env` if needed (e.g. a larger model like Qwen3-Embedding-0.6B, at the cost of much slower indexing for no real accuracy gain at this dataset size).

In [6]:
from build_index import build_index

build_index()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
embedding + indexing: 100%|██████████| 157/157 [02:04<00:00,  1.26it/s]

indexed 10002 products into collection 'amazon_products' at /Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/chroma_db


## 5. Sample hybrid queries

Semantic search + metadata filter, verified against the real index.

In [7]:
from retriever import RagRetriever, build_where

retriever = RagRetriever()
hits = retriever.search(
    "cozy throw blanket for couch",
    k=5,
    where=build_where(max_price=40),
)
for hit in hits:
    print(round(hit["score"], 3), "-", hit["title"], f"(${hit['price']})")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


0.624 - Ben&Jonah Throw Blankets, Perfect for The Fall & Winter. Fun Designs for Kids/Teens Great Present Friends and Family! 100% Soft Polyester. Camping/Travel Size (Rainbow Unicorn) ($24.45)
0.606 - Franco Kids Bedding Super Soft Plush Microfiber Blanket, Twin/Full Size 62" x 90", Fluffy Unicorn ($16.99)
0.565 - Disney Minnie Mouse Bowtique 'Garden Party' Fleece 62" x 90" Twin Blanket ($29.61)
0.555 - Toy Story My New Toys Woven Tapestry Throw Blanket ($33.99)
0.55 - Subrtex Throw Twin Warm Digital Printing All Season Blanket for Bed or Couch Super Soft (Boy and Puppy, 50''x60 ($28.62)


In [8]:
# Query with no filter, to sanity-check pure semantic ranking
for hit in retriever.search("birthday party balloons and decorations", k=5):
    print(round(hit["score"], 3), "-", hit["title"], f"(${hit['price']})")

0.698 - 18" Foil Rainbow Party Happy Birthday Balloon ($8.99)
0.652 - Conver USA 19083-18SP Get Well 3 Tulips Packed Balloon, 18", Multicolor ($2.45)
0.648 - Unique Party 56849 - 12" Latex Blush Pink Balloons, Pack of 50 ($9.3)
0.647 - Clear Latex Balloons Swirly Scrolls | Pack of 6 | Party Decor ($5.99)
0.647 - Creative Converting Hanging Décor New Year Countdown Banner ($9.79)


## What this index feeds

- **`rag.search` MCP tool** (`src/mcp_server/rag_tool.py`): wraps `RagRetriever.search()` directly, exposing the JSON schema `{query, k, max_price?, min_rating?, brand?, category?}` and building the Chroma `where` clause via `build_where(...)` internally. `category` filters on the `category_top_level` metadata written above.
- **Planner / Answerer agents** (`src/agents/`): cite results by `doc_id`, and every citation is verified against the evidence in code before it reaches the user. `brand`, `ingredients`, and `rating` are `None` for every product in this catalog, so the Answerer reports them as unavailable rather than filling them in — `brand_inferred` is available but is explicitly labeled a heuristic guess, never stated as fact.
- **`web.search` MCP tool + reconciliation** (`src/agents/retriever.py`): live results are matched back to these products by title similarity, since brand isn't available to match on. Live search only runs when the plan asks for current data or the LLM relevance check rejects the private hits.